In [ ]:
import os

# 1. Set your token environment variable
os.environ["KAGGLE_API_TOKEN"] = "KGAT_16772a49a145b93545707940cc4e3341"

# 2. Install kagglehub package
!pip install -q kagglehub

import kagglehub

# 3. Download the competition dataset
print("Downloading dataset using your token...")
path = kagglehub.competition_download('aptos2019-blindness-detection')

print("Dataset downloaded successfully to:", path)

100%|██████████| 9.51G/9.51G [09:18<00:00, 18.3MB/s]

Extracting files...
Extracting files...


Dataset downloaded successfully to: /root/.cache/kagglehub/competitions/aptos2019-blindness-detection
Dataset downloaded successfully to: /root/.cache/kagglehub/competitions/aptos2019-blindness-detection


In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
import kagglehub

# 1. Set your Kaggle API token and download dataset
os.environ["KAGGLE_API_TOKEN"] = "KGAT_16772a49a145b93545707940cc4e3341"
print("Downloading / loading dataset from cache...")
base_path = kagglehub.competition_download('aptos2019-blindness-detection')

csv_path = os.path.join(base_path, 'train.csv')
images_dir = os.path.join(base_path, 'train_images')

# 2. Organize images if not already done
if not os.path.exists('data/train'):
    print("Organizing dataset into train/val folders...")
    df = pd.read_csv(csv_path)
    df['file_name'] = df['id_code'].astype(str) + '.png'

    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['diagnosis'])

    def organize_images(subset_df, subset_name):
        target_base = f'data/{subset_name}'
        os.makedirs(target_base, exist_ok=True)
        for _, row in subset_df.iterrows():
            class_dir = os.path.join(target_base, str(row['diagnosis']))
            os.makedirs(class_dir, exist_ok=True)
            src = os.path.join(images_dir, row['file_name'])
            dst = os.path.join(class_dir, row['file_name'])
            if os.path.exists(src):
                shutil.copy(src, dst)

    organize_images(train_df, 'train')
    organize_images(val_df, 'val')
    print("Dataset organization complete!")
else:
    print("Dataset folders already exist. Skipping organization.")

Organizing dataset into train/val folders...
Dataset organization complete!


In [ ]:
# Install required libraries
!pip install -q timm tqdm scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

# Configurations
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-4
IMAGE_SIZE = 300
NUM_CLASSES = 5
MODEL_NAME = 'efficientnet_b3'

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Data Augmentations & Transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('data/train', transform=train_transforms)
val_dataset = datasets.ImageFolder('data/val', transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Load Pretrained Model
print(f"Loading {MODEL_NAME}...")
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Training Loop
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct.double() / len(val_dataset)

    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

    # Save best model weights
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        weight_filename = f"{MODEL_NAME}_dr_best.pth"
        torch.save(model.state_dict(), weight_filename)
        print(f"--> Saved Best Model to {weight_filename}")

print("Training finished successfully!")

Training on device: cuda
Loading efficientnet_b3...


model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch [1/5]: 100%|██████████| 184/184 [04:58<00:00,  1.62s/it, loss=0.355]


Val Loss: 0.8089 | Val Acc: 0.7558
--> Saved Best Model to efficientnet_b3_dr_best.pth


...

Val Loss: 0.6480 | Val Acc: 0.7954


Epoch [4/5]: 100%|██████████| 184/184 [04:54<00:00,  1.60s/it, loss=0.0853]


Val Loss: 0.6845 | Val Acc: 0.7940


Epoch [5/5]: 100%|██████████| 184/184 [04:54<00:00,  1.60s/it, loss=3.33]


Val Loss: 0.6683 | Val Acc: 0.7913
Training finished successfully!


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import timm
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# 1. Configurations
BATCH_SIZE = 16
IMAGE_SIZE = 300
NUM_CLASSES = 5
MODEL_NAME = 'efficientnet_b3'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

excel_path = 'test.csv'  # Update if your excel filename differs

# 2. Load the Excel/CSV File
if excel_path.endswith('.csv'):
    test_df = pd.read_csv(excel_path)
else:
    test_df = pd.read_excel(excel_path)

test_df['file_name'] = test_df['id_code'].astype(str) + '.png'

# 3. Universal Indexed Dataset (Recursively maps all image paths to prevent missing files)
class IndexedDataset(Dataset):
    def __init__(self, df, search_root='/content', transform=None):
        self.df = df
        self.transform = transform

        print(f"Indexing all images inside {search_root} (this takes a moment)...")
        self.file_path_map = {}
        for root, dirs, files in os.walk(search_root):
            for file in files:
                if file.endswith('.png') or file.endswith('.jpg'):
                    self.file_path_map[file] = os.path.join(root, file)
        print(f"Indexed {len(self.file_path_map)} images successfully!")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['file_name']
        label = int(row['diagnosis'])

        img_path = self.file_path_map.get(img_name)
        if not img_path or not os.path.exists(img_path):
            raise FileNotFoundError(f"Image '{img_name}' could not be found anywhere in the workspace directory index!")

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = IndexedDataset(test_df, search_root='/content', transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 4. Load Trained Model Weights
weight_path = f"{MODEL_NAME}_dr_best.pth"
print(f"Loading weights from {weight_path}...")

model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# 5. Run Evaluation Loop
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating on Test Set"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 6. Calculate Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

print("\n================ TEST SET METRICS ================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("==================================================\n")

class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
print("Detailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Indexing all images inside /content (this takes a moment)...
Indexed 3662 images successfully!
Loading weights from efficientnet_b3_dr_best.pth...


Evaluating on Test Set: 100%|██████████| 23/23 [00:34<00:00,  1.52s/it]


================ TEST SET METRICS ================
Accuracy  : 86.07%
Precision : 86.09%
Recall    : 86.07%
F1-Score  : 85.80%

Detailed Classification Report:
                   precision    recall  f1-score   support

        No DR (0)       0.97      0.99      0.98       199
         Mild (1)       0.62      0.60      0.61        30
     Moderate (2)       0.78      0.84      0.81        87
       Severe (3)       0.47      0.53      0.50        17
Proliferative (4)       0.82      0.55      0.65        33

         accuracy                           0.86       366
        macro avg       0.73      0.70      0.71       366
     weighted avg       0.86      0.86      0.86       366


Confusion Matrix:
[[197   1   1   0   0]
 [  4  18   6   1   1]
 [  2   6  73   5   1]
 [  0   0   6   9   2]
 [  0   4   7   4  18]]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from tqdm import tqdm

# 1. Configurations
MODEL_NAME = 'efficientnet_b3'
NUM_CLASSES = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FINE_TUNE_EPOCHS = 4
FINE_TUNE_LR = 1e-5  # Low learning rate for delicate weight adjustments

# 2. Load Existing Best Weights
weight_path = f"{MODEL_NAME}_dr_best.pth"
print(f"Loading existing weights from {weight_path}...")

model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
model = model.to(DEVICE)

# 3. Advanced Strategy: Selective Layer Freezing
# Freeze early blocks to retain low-level edge/texture knowledge
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the final classification head
for param in model.get_classifier().parameters():
    param.requires_grad = True

# Unfreeze the deeper blocks (blocks 4 and 5 in EfficientNet-B3) for high-level feature refinement
for name, param in model.named_parameters():
    if 'blocks.4' in name or 'blocks.5' in name:
        param.requires_grad = True

print("Selective freezing applied: Only deeper blocks and classifier head are trainable.")

# 4. Advanced Strategy: Class Weights for Imbalanced Medical Data
# Giving higher loss penalties for misclassifying minority classes (Mild: 1, Severe: 3, Proliferative: 4)
# (Adjust these weights based on your training label distribution if needed)
class_weights = torch.tensor([1.0, 3.2, 1.5, 3.8, 2.8], dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# 5. Optimizer & Scheduler (Only optimizing parameters that require gradients)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=FINE_TUNE_LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINE_TUNE_EPOCHS)

print(f"\nStarting Advanced Fine-Tuning for {FINE_TUNE_EPOCHS} epochs...")
best_val_loss = float('inf')

for epoch in range(FINE_TUNE_EPOCHS):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Fine-Tune Epoch [{epoch+1}/{FINE_TUNE_EPOCHS}]")
    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data)
        total += labels.size(0)

        loop.set_postfix(loss=loss.item(), acc=(correct.double() / total).item())

    scheduler.step()

    # Validation Loop
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct.double() / len(val_dataset)

    print(f"Epoch {epoch+1} Results -> Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc * 100:.2f}%")

    # Save the improved model if validation loss decreases
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), weight_path)
        print(f"--> Saved Improved Fine-Tuned Model to {weight_path}!\n")

print("Advanced fine-tuning complete! Re-run your test evaluation script to check your updated performance metrics.")

Loading existing weights from efficientnet_b3_dr_best.pth...
Selective freezing applied: Only deeper blocks and classifier head are trainable.

Starting Advanced Fine-Tuning for 4 epochs...


Fine-Tune Epoch [1/4]: 100%|██████████| 184/184 [04:47<00:00,  1.56s/it, acc=0.86, loss=1.15]


Epoch 1 Results -> Val Loss: 0.6585 | Val Acc: 78.31%
--> Saved Improved Fine-Tuned Model to efficientnet_b3_dr_best.pth!



Fine-Tune Epoch [2/4]: 100%|██████████| 184/184 [04:48<00:00,  1.57s/it, acc=0.873, loss=0.615]


Epoch 2 Results -> Val Loss: 0.6278 | Val Acc: 78.44%
--> Saved Improved Fine-Tuned Model to efficientnet_b3_dr_best.pth!



Fine-Tune Epoch [3/4]: 100%|██████████| 184/184 [04:48<00:00,  1.57s/it, acc=0.876, loss=4.25]


Epoch 3 Results -> Val Loss: 0.6239 | Val Acc: 79.13%
--> Saved Improved Fine-Tuned Model to efficientnet_b3_dr_best.pth!



Fine-Tune Epoch [4/4]: 100%|██████████| 184/184 [04:48<00:00,  1.57s/it, acc=0.883, loss=0.474]


Epoch 4 Results -> Val Loss: 0.6210 | Val Acc: 78.58%
--> Saved Improved Fine-Tuned Model to efficientnet_b3_dr_best.pth!

Advanced fine-tuning complete! Re-run your test evaluation script to check your updated performance metrics.


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import timm
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# 1. Configurations
BATCH_SIZE = 16
IMAGE_SIZE = 300
NUM_CLASSES = 5
MODEL_NAME = 'efficientnet_b3'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

excel_path = 'test.csv'  # Update if your excel filename differs

# 2. Load the Excel/CSV File
if excel_path.endswith('.csv'):
    test_df = pd.read_csv(excel_path)
else:
    test_df = pd.read_excel(excel_path)

test_df['file_name'] = test_df['id_code'].astype(str) + '.png'

# 3. Universal Indexed Dataset (Recursively maps all image paths to prevent missing files)
class IndexedDataset(Dataset):
    def __init__(self, df, search_root='/content', transform=None):
        self.df = df
        self.transform = transform

        print(f"Indexing all images inside {search_root} (this takes a moment)...")
        self.file_path_map = {}
        for root, dirs, files in os.walk(search_root):
            for file in files:
                if file.endswith('.png') or file.endswith('.jpg'):
                    self.file_path_map[file] = os.path.join(root, file)
        print(f"Indexed {len(self.file_path_map)} images successfully!")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['file_name']
        label = int(row['diagnosis'])

        img_path = self.file_path_map.get(img_name)
        if not img_path or not os.path.exists(img_path):
            raise FileNotFoundError(f"Image '{img_name}' could not be found anywhere in the workspace directory index!")

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = IndexedDataset(test_df, search_root='/content', transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 4. Load Trained Model Weights
weight_path = f"{MODEL_NAME}_dr_best.pth"
print(f"Loading weights from {weight_path}...")

model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# 5. Run Evaluation Loop
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating on Test Set"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 6. Calculate Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

print("\n================ TEST SET METRICS ================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("==================================================\n")

class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
print("Detailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Indexing all images inside /content (this takes a moment)...
Indexed 3662 images successfully!
Loading weights from efficientnet_b3_dr_best.pth...


Evaluating on Test Set: 100%|██████████| 23/23 [00:34<00:00,  1.51s/it]


================ TEST SET METRICS ================
Accuracy  : 86.89%
Precision : 87.25%
Recall    : 86.89%
F1-Score  : 86.67%

Detailed Classification Report:
                   precision    recall  f1-score   support

        No DR (0)       0.98      0.98      0.98       199
         Mild (1)       0.62      0.50      0.56        30
     Moderate (2)       0.77      0.89      0.82        87
       Severe (3)       0.52      0.65      0.58        17
Proliferative (4)       0.87      0.61      0.71        33

         accuracy                           0.87       366
        macro avg       0.75      0.72      0.73       366
     weighted avg       0.87      0.87      0.87       366


Confusion Matrix:
[[195   3   1   0   0]
 [  3  15  10   1   1]
 [  0   3  77   6   1]
 [  0   0   5  11   1]
 [  0   3   7   3  20]]


In [ ]:
import os

print("Checking Kaggle API credentials...")
if os.path.exists('/root/.kaggle/kaggle.json'):
    print("kaggle.json found in /root/.kaggle/")
else:
    print("WARNING: kaggle.json NOT found! Please upload your kaggle.json token first.")

print("\nListing contents of /content/idrid_dataset:")
if os.path.exists('/content/idrid_dataset'):
    for root, dirs, files in os.walk('/content/idrid_dataset'):
        print(f"Directory: {root}")
        print(f"Files: {files[:5]} (showing up to 5)")
else:
    print("/content/idrid_dataset directory does not exist yet.")

Checking Kaggle API credentials...

Listing contents of /content/idrid_dataset:
Directory: /content/idrid_dataset
Files: ['idrid_labels.csv'] (showing up to 5)
Directory: /content/idrid_dataset/Imagenes
Files: [] (showing up to 5)
Directory: /content/idrid_dataset/Imagenes/Imagenes
Files: ['IDRiD_314.jpg', 'IDRiD_233.jpg', 'IDRiD_008.jpg', 'IDRiD_135.jpg', 'IDRiD_063.jpg'] (showing up to 5)


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import timm
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# 1. Exact Paths Discovered
csv_path = '/content/idrid_dataset/idrid_labels.csv'
imgs_dir = '/content/idrid_dataset/Imagenes/Imagenes'

print(f"Loading labels from: {csv_path}")
print(f"Loading images from: {imgs_dir}")

# 2. Configurations
BATCH_SIZE = 16
IMAGE_SIZE = 300
NUM_CLASSES = 5
MODEL_NAME = 'efficientnet_b3'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Load Dataframe and Detect Column Names
test_df = pd.read_csv(csv_path)
print("CSV Columns found:", test_df.columns.tolist())

# Automatically map column names based on common naming conventions
image_col = next((col for col in ['Image name', 'id_code', 'image', 'file_name'] if col in test_df.columns), test_df.columns[0])
label_col = next((col for col in ['Retinopathy grade', 'diagnosis', 'label', 'grade'] if col in test_df.columns), test_df.columns[1])

print(f"Mapped Image Column: '{image_col}' | Mapped Label Column: '{label_col}'")

test_df['file_name'] = test_df[image_col].astype(str)
if not test_df['file_name'].iloc[0].endswith(('.jpg', '.png')):
    test_df['file_name'] += '.jpg'  # Append extension if missing

test_df['diagnosis'] = test_df[label_col].astype(int)

# 4. Custom Dataset for IDRiD Local Testing
class LocalIDRiDDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['file_name']
        label = int(row['diagnosis'])

        img_path = os.path.join(self.img_dir, img_name)

        # Fallback if filename has a different casing or extension
        if not os.path.exists(img_path):
            # Try png instead of jpg or vice versa
            alt_path = os.path.join(self.img_dir, img_name.replace('.jpg', '.png'))
            if os.path.exists(alt_path):
                img_path = alt_path
            else:
                raise FileNotFoundError(f"Could not find image {img_name} in {self.img_dir}")

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = LocalIDRiDDataset(test_df, imgs_dir, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 5. Load Trained Model Weights
weight_path = f"{MODEL_NAME}_dr_best.pth"
print(f"Loading model weights from {weight_path}...")

model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# 6. Run Evaluation Loop
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating on IDRiD Dataset"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 7. Calculate Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

print("\n================ IDRiD TEST SET METRICS ================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("========================================================\n")

class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
print("Detailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Loading labels from: /content/idrid_dataset/idrid_labels.csv
Loading images from: /content/idrid_dataset/Imagenes/Imagenes
CSV Columns found: ['id_code', 'diagnosis', 'Risk of macular edema ', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
Mapped Image Column: 'id_code' | Mapped Label Column: 'diagnosis'
Loading model weights from efficientnet_b3_dr_best.pth...


Evaluating on IDRiD Dataset: 100%|██████████| 29/29 [00:49<00:00,  1.71s/it]


================ IDRiD TEST SET METRICS ================
Accuracy  : 54.29%
Precision : 59.01%
Recall    : 54.29%
F1-Score  : 54.64%

Detailed Classification Report:
                   precision    recall  f1-score   support

        No DR (0)       0.86      0.66      0.75       129
         Mild (1)       0.16      0.55      0.25        22
     Moderate (2)       0.57      0.72      0.64       156
       Severe (3)       0.41      0.25      0.31        84
Proliferative (4)       0.47      0.25      0.33        64

         accuracy                           0.54       455
        macro avg       0.50      0.49      0.45       455
     weighted avg       0.59      0.54      0.55       455


Confusion Matrix:
[[ 85  33  11   0   0]
 [  6  12   4   0   0]
 [  7  23 113   5   8]
 [  1   3  49  21  10]
 [  0   3  20  25  16]]


In [ ]:
# Install required libraries
!pip install -q timm tqdm scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

# Configurations
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-4
IMAGE_SIZE = 300
NUM_CLASSES = 5
MODEL_NAME = 'efficientnet_b3'

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Data Augmentations & Transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('data/train', transform=train_transforms)
val_dataset = datasets.ImageFolder('data/val', transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Define Hybrid CNN-Mamba Model
class HybridEfficientNetMamba(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridEfficientNetMamba, self).__init__()

        # Extract features using timm backbone
        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, features_only=True)

        # Dynamically get final channel count
        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        features = self.backbone(dummy_input)
        final_channels = features[-1].shape[1]

        # Projection layer to reduce channels
        self.proj = nn.Conv2d(final_channels, 256, kernel_size=1)

        # State-space sequence mixing components
        self.depthwise_conv = nn.Conv1d(256, 256, kernel_size=3, padding=1, groups=256)
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.2)

        # Classification Head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # 1. Extract features from CNN backbone
        feat_list = self.backbone(x)
        x = feat_list[-1] # Shape: [Batch, Channels, Height, Width]

        # 2. Reduce channels
        x = self.proj(x)
        b, c, h, w = x.shape

        # 3. Flatten 2D spatial grid into a 1D sequence: [B, Channels, Height * Width]
        x = x.view(b, c, h * w)
        x_res = x

        # 4. Pass through sequence mixer keeping [B, C, L] format for Conv1d
        x = self.depthwise_conv(x)
        x = torch.nn.functional.silu(x)

        # Permute to [B, L, C] for Linear projection layers
        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        # Permute back to [B, C, L] for residual connection
        x = x.permute(0, 2, 1)
        x = x + x_res

        # 5. Global Pooling and Classify
        out = self.classifier(x)
        return out

# Load Hybrid Model
print(f"Loading Hybrid CNN-Mamba Model...")
model = HybridEfficientNetMamba(num_classes=NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Training Loop
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct.double() / len(val_dataset)

    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

    # Save best model weights with a unique name (will not overwrite baseline)
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        weight_filename = "hybrid_mamba_dr_best.pth"
        torch.save(model.state_dict(), weight_filename)
        print(f"--> Saved Best Hybrid Model to {weight_filename}")

print("Hybrid Training finished successfully!")

Training on device: cuda
Loading Hybrid CNN-Mamba Model...


Epoch [1/5]: 100%|██████████| 184/184 [04:43<00:00,  1.54s/it, loss=1.71]


Val Loss: 0.5275 | Val Acc: 0.8104
--> Saved Best Hybrid Model to hybrid_mamba_dr_best.pth


Epoch [2/5]: 100%|██████████| 184/184 [04:46<00:00,  1.56s/it, loss=2.08]


Val Loss: 0.4993 | Val Acc: 0.7981
--> Saved Best Hybrid Model to hybrid_mamba_dr_best.pth


...

Val Loss: 0.5312 | Val Acc: 0.8145


Epoch [4/5]: 100%|██████████| 184/184 [04:43<00:00,  1.54s/it, loss=2.2]


Val Loss: 0.4997 | Val Acc: 0.8281


Epoch [5/5]: 100%|██████████| 184/184 [04:42<00:00,  1.54s/it, loss=0.581]


Val Loss: 0.6149 | Val Acc: 0.8267
Hybrid Training finished successfully!


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import timm
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# 1. Configurations
BATCH_SIZE = 16
IMAGE_SIZE = 300
NUM_CLASSES = 5
WEIGHT_PATH = "hybrid_mamba_dr_best.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

excel_path = 'test.csv'  # Update if your excel filename differs

# 2. Load the Excel/CSV File
if excel_path.endswith('.csv'):
    test_df = pd.read_csv(excel_path)
else:
    test_df = pd.read_excel(excel_path)

test_df['file_name'] = test_df['id_code'].astype(str) + '.png'

# 3. Universal Indexed Dataset
class IndexedDataset(Dataset):
    def __init__(self, df, search_root='/content', transform=None):
        self.df = df
        self.transform = transform

        print(f"Indexing all images inside {search_root} (this takes a moment)...")
        self.file_path_map = {}
        for root, dirs, files in os.walk(search_root):
            for file in files:
                if file.endswith('.png') or file.endswith('.jpg'):
                    self.file_path_map[file] = os.path.join(root, file)
        print(f"Indexed {len(self.file_path_map)} images successfully!")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['file_name']
        label = int(row['diagnosis'])

        img_path = self.file_path_map.get(img_name)
        if not img_path or not os.path.exists(img_path):
            raise FileNotFoundError(f"Image '{img_name}' could not be found anywhere in the workspace directory index!")

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = IndexedDataset(test_df, search_root='/content', transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 4. Define the Custom Hybrid CNN-Mamba Architecture Class
class HybridEfficientNetMamba(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridEfficientNetMamba, self).__init__()

        self.backbone = timm.create_model('efficientnet_b3', pretrained=False, features_only=True)

        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        features = self.backbone(dummy_input)
        final_channels = features[-1].shape[1]

        self.proj = nn.Conv2d(final_channels, 256, kernel_size=1)
        self.depthwise_conv = nn.Conv1d(256, 256, kernel_size=3, padding=1, groups=256)
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.2)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feat_list = self.backbone(x)
        x = feat_list[-1]

        x = self.proj(x)
        b, c, h, w = x.shape

        x = x.view(b, c, h * w)
        x_res = x

        x = self.depthwise_conv(x)
        x = torch.nn.functional.silu(x)

        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        x = x.permute(0, 2, 1)
        x = x + x_res

        out = self.classifier(x)
        return out

# 5. Instantiate Model and Load Saved Weights
print(f"Loading weights from {WEIGHT_PATH}...")
model = HybridEfficientNetMamba(num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(WEIGHT_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# 6. Run Evaluation Loop
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating on Test Set"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 7. Calculate Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

print("\n================ TEST SET METRICS ================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("==================================================\n")

class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
print("Detailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Indexing all images inside /content (this takes a moment)...
Indexed 4117 images successfully!
Loading weights from hybrid_mamba_dr_best.pth...


Evaluating on Test Set: 100%|██████████| 23/23 [00:33<00:00,  1.47s/it]


================ TEST SET METRICS ================
Accuracy  : 86.61%
Precision : 87.53%
Recall    : 86.61%
F1-Score  : 85.86%

Detailed Classification Report:
                   precision    recall  f1-score   support

        No DR (0)       0.98      0.98      0.98       199
         Mild (1)       0.70      0.53      0.60        30
     Moderate (2)       0.70      0.93      0.80        87
       Severe (3)       0.60      0.35      0.44        17
Proliferative (4)       1.00      0.55      0.71        33

         accuracy                           0.87       366
        macro avg       0.80      0.67      0.71       366
     weighted avg       0.88      0.87      0.86       366


Confusion Matrix:
[[196   2   1   0   0]
 [  3  16  11   0   0]
 [  1   3  81   2   0]
 [  0   0  11   6   0]
 [  0   2  11   2  18]]


In [ ]:
# Install required libraries
!pip install -q timm tqdm scikit-learn

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

# 1. Upgraded Configurations
BATCH_SIZE = 16
EPOCHS = 15  # Increased epochs; early stopping will prevent overfitting
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2  # Stronger weight decay for better generalization
IMAGE_SIZE = 300
NUM_CLASSES = 5
MODEL_NAME = 'efficientnet_b3'
WEIGHT_FILENAME = "hybrid_mamba_improved_best.pth"  # New unique save file

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Advanced Data Augmentations (Crucial for medical imaging generalization)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1), # Handles varying clinic lighting
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)), # Zoom & shift invariance
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('data/train', transform=train_transforms)
val_dataset = datasets.ImageFolder('data/val', transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 3. Define Upgraded Hybrid CNN-Mamba Model with Enhanced Dropout
class HybridEfficientNetMambaUpgraded(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridEfficientNetMambaUpgraded, self).__init__()

        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, features_only=True)

        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        features = self.backbone(dummy_input)
        final_channels = features[-1].shape[1]

        self.proj = nn.Conv2d(final_channels, 256, kernel_size=1)

        # Sequence mixing components with balanced dropout
        self.depthwise_conv = nn.Conv1d(256, 256, kernel_size=3, padding=1, groups=256)
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.3)  # Increased dropout for regularization

        # Classification Head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feat_list = self.backbone(x)
        x = feat_list[-1]

        x = self.proj(x)
        b, c, h, w = x.shape

        x = x.view(b, c, h * w)
        x_res = x

        x = self.depthwise_conv(x)
        x = torch.nn.functional.silu(x)

        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        x = x.permute(0, 2, 1)
        x = x + x_res

        out = self.classifier(x)
        return out

# Initialize Model, Optimizer, Scheduler, and Early Stopping
model = HybridEfficientNetMambaUpgraded(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

class EarlyStopping:
    def __init__(self, patience=4, delta=0.0005):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False

early_stopping = EarlyStopping(patience=4)
best_val_loss = float('inf')

print(f"\n--- Starting Upgraded Hybrid Training (Saving to: {WEIGHT_FILENAME}) ---")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct.double() / len(val_dataset)

    # Step the learning rate scheduler
    scheduler.step()

    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    # Check if this is the best model based on validation loss
    if early_stopping(epoch_val_loss):
        torch.save(model.state_dict(), WEIGHT_FILENAME)
        print(f"--> Saved Improved Best Model to {WEIGHT_FILENAME}")

    if early_stopping.early_stop:
        print("\nEarly stopping triggered! Training halted to prevent overfitting.")
        break

print("Training finished successfully! You can now evaluate your new model using 'hybrid_mamba_improved_best.pth'.")

Training on device: cuda



--- Starting Upgraded Hybrid Training (Saving to: hybrid_mamba_improved_best.pth) ---


Epoch [1/15]: 100%|██████████| 184/184 [04:51<00:00,  1.58s/it, loss=1.7]


Val Loss: 0.5760 | Val Acc: 0.7790 | LR: 0.000099
--> Saved Improved Best Model to hybrid_mamba_improved_best.pth


Epoch [2/15]: 100%|██████████| 184/184 [04:53<00:00,  1.59s/it, loss=0.344]


Val Loss: 0.5357 | Val Acc: 0.7926 | LR: 0.000096
--> Saved Improved Best Model to hybrid_mamba_improved_best.pth


Epoch [3/15]:  47%|████▋     | 86/184 [02:20<02:11,  1.34s/it, loss=0.674]